In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time
import joblib
 
# Preprocessing
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer
 
# Models
from sklearn.linear_model import Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.neural_network import MLPRegressor
 
# Model selection
from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    KFold,
)
 
# Metrics
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
 
# Feature importance
from sklearn.inspection import permutation_importance


In [2]:
CONFIG = {
    "input": "Outputs/lr_epc_towns.parquet",
    "test_size": 0.2,
    "random_state": 42,
    "cv_folds": 5,
    "n_jobs": -1,
 
    # Columns to drop (merge keys, address fields, intermediate columns)
    "drop_columns": [
        "transaction_id",       # Identifier

        "postcode_x",           # Postcodes — five versions
        "postcode_clean", 
        "postcode_merge",
        "PCDS", 
        "postcode_y",

        "address_key",          # Address components 
        "address1",
        "paon", 
        "saon", 
        "street", 
        "locality", 
        "town", 
        "district", 
        "county",

        "record_status",        # Admin fields -> Land Registry internal codes, not property attributes
        "ppd_category",

                                # Pipeline fields — served their purpose during data joining
        "uprn",                 # used to get exact coordinates
        "lodgement_date",       # EPC lodgement date, not the sale date
        
                                # Coordinate fallbacks — keeping exact_lat/exact_lon, dropping centroid
        "LAT",                  # postcode centroid latitude (less precise)
        "LONG",                 # postcode centroid longitude (less precise)
        "has_exact_coords",     # flag column, not a feature
        
        "dist_bus_km",          # Bus Features -> I dont want them right now
        "bus_within_1km",

        "nearest_town",         # Town name — keeping only dist_town_km, since the name is one of 112 categories
    ],
}

In [ ]:
print("Loading dataset...")
df = pd.read_parquet(CONFIG["input"])
print(f"  Raw: {len(df):,} rows, {len(df.columns)} columns")

# Drop rows with no EPC (no house details)
before = len(df)
df = df[df["total_floor_area"].notna()]
print(f"  Has EPC: {before:,} → {len(df):,} (dropped {before - len(df):,})")

# Drop rows with no coordinates (those have no distances)
before = len(df)
df = df[df["dist_town_km"].notna()]
print(f"  Has coords: {before:,} → {len(df):,} (dropped {before - len(df):,})")

# Drop useless columns
df = df.drop(columns=CONFIG["drop_columns"], errors="ignore")

# Extract date features
df["sale_year"] = df["date"].dt.year
df["sale_month"] = df["date"].dt.month
df = df.drop(columns=["date"])

# Clean up construction_age_band to remove "England and Wales: " prefix, if present
df["construction_age_band"] = df["construction_age_band"].str.replace("England and Wales: ", "", regex=False)

print(f"  After filtering: {len(df):,} rows, {len(df.columns)} columns")

Loading dataset...
  Raw: 5,890,089 rows, 53 columns
  Has EPC: 5,890,089 → 4,379,246 (dropped 1,510,843)
  Has coords: 4,379,246 → 4,379,226 (dropped 20)
  After filtering: 4,379,226 rows, 29 columns


In [4]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 50)
pd.set_option("display.width", None)
print(df.head().to_string())

     price property_type_x new_build duration  total_floor_area  current_energy_efficiency current_energy_rating  number_habitable_rooms            tenure property_type_y transaction_type           construction_age_band   built_form                  main_fuel mains_gas_flag  exact_lat  exact_lon  dist_primary_km  dist_secondary_km  dist_rail_km  rail_within_1km  rail_within_5km  dist_metro_km  metro_within_1km  dist_airport_km  dist_coast_km  dist_town_km  sale_year  sale_month
0  1281150               T         N        F             164.0                       65.0                     D                     7.0    owner-occupied           House    Marketed sale    England and Wales: 1900-1929  Mid-Terrace  mains gas (not community)              Y  51.552537  -0.107757         0.177971           0.431792      0.157559              2.0             46.0       0.379762               4.0        43.869898       4.633338     14.663999       2023           6
1  1675000               T        

In [ ]:
# =============================================================================
# SECTION 2: DEFINE FEATURES AND TARGET
# =============================================================================
 
target = "price"
 
numeric_features = [
    # EPC features
    "total_floor_area",           # floor area in m²
    "current_energy_efficiency",  # numeric efficiency score
    "number_habitable_rooms",     # number of rooms
 
    # Location — exact coordinates
    "exact_lat",                  # property latitude
    "exact_lon",                  # property longitude
 
    # Spatial — school distances
    "dist_primary_km",            # distance to nearest primary school
    "dist_secondary_km",          # distance to nearest secondary school
 
    # Spatial — transport distances and density
    "dist_rail_km",               # distance to nearest rail station
    "rail_within_1km",            # number of rail stations within 1km
    "rail_within_5km",            # number of rail stations within 5km
    "dist_metro_km",              # distance to nearest metro/underground
    "metro_within_1km",           # number of metro stations within 1km
    "dist_airport_km",            # distance to nearest airport
 
    # Spatial — geography
    "dist_coast_km",              # distance to nearest coastline
    "dist_town_km",               # distance to nearest major town (ONS 75k+)
 
    # Date
    "sale_year",                  # year of sale
    "sale_month",                 # month of sale
]
 
categorical_features = [
    "property_type_x",            # D=Detached, S=Semi, T=Terraced, F=Flat, O=Other
    "duration",                   # F=Freehold, L=Leasehold
    "current_energy_rating",      # A-G energy rating
    "tenure",                     # owner-occupied, rented, etc.
    "property_type_y",            # House, Flat, Bungalow (from EPC)
    "built_form",                 # Detached, Semi-Detached, Mid-Terrace, etc.
    "construction_age_band",      # e.g. England and Wales: 1900-1929
    "main_fuel",                  # mains gas, electricity, etc.
    "transaction_type",           # marketed sale, rental, etc.
]
 
binary_features = [
    "new_build",                  # Y/N — is it a new build
    "mains_gas_flag",             # Y/N — connected to mains gas
]